# 🖼️ Pipeline: CNN para Clasificación de Imágenes

**Propósito:** Pipeline completo de Visión por Computadora — desde la carga de datos
hasta la evaluación, incluyendo Transfer Learning y Data Augmentation.

---
## 📋 Flujo de trabajo

1. Cargar dataset (Fashion MNIST)
2. Preprocesar (normalizar, reshape)
3. Crear CNN (`crear_cnn()`)
4. Entrenar (`compilar_y_entrenar()`)
5. Evaluar (`evaluar_clasificacion()`)
6. Data Augmentation
7. Transfer Learning (comentado)
8. Guardar modelo

In [ ]:
# MACHOTE — Setup universal
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from modulo4_libreria import *

INFO = setup_completo()

In [ ]:
# MACHOTE — Cargar Fashion MNIST

print("Cargando Fashion MNIST...")
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

CLASES = ['Camiseta', 'Pantalón', 'Pullover', 'Vestido', 'Abrigo',
          'Sandalia', 'Camisa', 'Zapatilla', 'Bolsa', 'Bota']

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Clases: {len(CLASES)}")

mostrar_lote_imagenes(X_train, y_train, clases=CLASES, n=10)

In [ ]:
# MACHOTE — Preprocesamiento

# Normalizar a [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# Reshape para CNN: (28, 28) → (28, 28, 1)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# Separar validation set del train
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Rango de píxeles: [{X_train.min():.2f}, {X_train.max():.2f}]")

In [ ]:
# MACHOTE — Crear CNN

cnn = crear_cnn(input_shape=(28,28,1), num_clases=10, filtros=[32, 64, 128])
cnn.summary()

In [ ]:
# MACHOTE — Entrenar CNN

hist = compilar_y_entrenar(
    cnn,
    X_train, y_train,
    X_val, y_val,
    num_clases=10,
    epochs=30,
    batch_size=64,
    early_stop_paciencia=5
)

graficar_historia(hist, 'CNN — Fashion MNIST')

In [ ]:
# MACHOTE — Evaluar en test

preds = evaluar_clasificacion(cnn, X_test, y_test, nombres_clases=CLASES)

In [ ]:
# MACHOTE — Guardar modelo entrenado

ruta = guardar_modelo(cnn, 'cnn_fashion_mnist', ruta_modelos=INFO['rutas']['modelos'])
print(f"Modelo guardado en: {ruta}")

---
## 🧪 Data Augmentation

Aumentamos la variabilidad del dataset aplicando transformaciones aleatorias.
Esto reduce overfitting y mejora la generalización.

In [ ]:
# MACHOTE — Data Augmentation

data_aug = crear_data_augmentation()

# Visualizar aumentos
plt.figure(figsize=(12, 4))
for i in range(8):
    img_aug = data_aug(X_train[:1])[0].numpy()
    plt.subplot(2, 4, i+1)
    plt.imshow(img_aug.squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0:
        plt.title('Original + aumentos')
plt.suptitle('Data Augmentation — Fashion MNIST', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Data Augmentation pipeline lista para usar en entrenamiento.")

---
## 🔁 Transfer Learning (Ejemplo conceptual)

Para datasets más grandes (>1000 imágenes por clase) se recomienda usar
una base preentrenada en ImageNet (MobileNetV2, VGG16, ResNet50, etc.).

> **Nota:** Fashion MNIST tiene imágenes de 28×28 en 1 canal. Transfer Learning requiere
> 224×224×3. Para usar transfer learning habría que redimensionar y replicar canales.
> El siguiente código es ilustrativo.

In [ ]:
# MACHOTE — Transfer Learning (concepto / requiere redimensionar)
#
# Para usar en un dataset propio con imágenes grandes:
#
# from tensorflow.keras.applications import MobileNetV2
# from tensorflow.keras import layers, models
#
# modelo_tl, base = crear_transfer_learning(
#     num_clases=10,
#     input_shape=(224, 224, 3),
#     base='MobileNetV2',
#     trainable_base=False  # True para fine-tuning
# )
#
# modelo_tl.summary()
#
# # Compilar y entrenar igual que cualquier modelo:
# # hist_tl = compilar_y_entrenar(modelo_tl, ...)

print("✅ Código de Transfer Learning disponible en crear_transfer_learning()")
print("   Ver documentación: help(crear_transfer_learning)")

---
## ⚠️ Problemas comunes y soluciones

| Problema | Síntoma | Solución |
|----------|---------|----------|
| **Overfitting** | Train acc alta, val baja | Aumentar Dropout, reducir capas, añadir Data Augmentation |
| **Underfitting** | Train acc baja | Más capas/filtros, más épocas, menor dropout |
| **Gradiente explosivo** | Loss = NaN | Reducir learning rate, añadir BatchNormalization |
| **Datos desbalanceados** | Clase minoritaria ignorada | Weighted loss, oversampling |
| **Memoria GPU insuficiente** | OOM error | Reducir batch_size, usar imágenes más pequeñas |
| **No converge** | Loss oscila | Reducir lr, usar Adam en vez de SGD |

---
*Pipeline creado para el Diplomado en Redes Neuronales — Módulo 4* 🧠

In [ ]:
# TODO: Experimenta con estas variaciones
# 1. Cambia filtros=[64, 128, 256] — ¿mejora?
# 2. Usa kernel_size=5 en lugar de 3
# 3. Entrena con Data Augmentation incluida en el modelo
# 4. Prueba con CIFAR-10 en lugar de Fashion MNIST
#    (solo cambia el dataset, la CNN funciona igual con 3 canales)